# Notebook for Journeylenz batch run

1. Instrucitons on how to run are listed below. Please read in order.
2. If you want to do a fresh run, delete the checkpiont table from databricks. checkpoint table name can be found in cell 24.
3. The model workflow accepts a pandas df. If you want to only run on a sample of the input table, call a `await timed_run(df.head(100).copy())` as opposed to `await timed_run(df.head.copy())` to test in cell 23
4. Run cells in order. First cells handle model gateway auth, then to load the datatset and handle some minor pre-processing, then python code to set-up gateway inference and batch run, finally, there is code to either build an ouput dataset from the checkpoint (that is created intermittently during the batch run), or to buid an output dataset from the in-memory table.
5. The model will output progress as it runs inference. On a big run, it can happen that auth needs to be handled again, the model will do this automatically for each thread. It could be it reaches token limit, in which case it will pause and start again when token limit is reset. It could be the workflow breaks due to cluster outage (must run on UC Cluster for model gateway access). That is the purpose of the chekpoint, so the code will check which conversationid's are in the checkepoint table, drop those from the input table and continue inference on all those that are left. It will then continue intermittently saving to checkpoint table.
6. the workflow produces two outputs. One initial output and one flattened output. This needs to be done manually after the model inference is complete. Intial output = 1 row/trasncript. Flattened output = 1 row/topic in that transcript. The flattened output is more useful for downstream as it shows individual topic view rather than combined topics within a single conversation.
7. I have found if it is the first time i'm running the model gateway workflow in a while, it takes a long time. It seems to need to 'wake-up' so always good to do a first run on a few samples and then do a run on a big sample if needed

# Model Gateway

In [0]:
%pip install networkx
%pip install azure.keyvault==4.2.0 --quiet
%pip install msal==1.34.0 --quiet
%pip install rapidfuzz
%restart_python

In [0]:
import re
import ast
import time
import json
import html
import requests

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import pyspark.sql.functions as F

from matplotlib.patches import Patch
from typing import Dict, Any, List, Tuple, Optional
from azure.keyvault.secrets import SecretClient 
from msal import ConfidentialClientApplication
from rapidfuzz.fuzz import partial_ratio
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
def get_uc_secrets_map(connector: str, 
                       vault_url: str, 
                       *secret_keys: str) -> dict[str, str]:
    credential = dbutils.credentials.getServiceCredentialsProvider(connector)
    client = SecretClient(vault_url=vault_url, credential=credential)
    return {k: client.get_secret(k).value for k in secret_keys}

def get_oauth_token(tenant_id: str, 
                    client_id: str, 
                    client_secret: str, 
                    scopes: list[str]) -> str:
    app = ConfidentialClientApplication(client_id, client_credential=client_secret, authority=f"https://login.microsoftonline.com/{tenant_id}")
    result = app.acquire_token_for_client(scopes=scopes)    
    if "access_token" in result:
        return result["access_token"]

def post_with_bearer(api_url: str, 
                     token: str, 
                     json_payload: dict, 
                     verify: str | bool = True, 
                     timeout: int = 60) -> requests.Response:
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
    resp = requests.post(api_url, headers=headers, json=json_payload, verify=verify, timeout=timeout)
    resp.raise_for_status()
    return resp

def get_with_bearer(api_url: str, 
                    token: str, 
                    verify: str | bool = True, 
                    timeout: int = 60) -> requests.Response:
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
    resp = requests.get(api_url, headers=headers, verify=verify, timeout=timeout)
    resp.raise_for_status()
    return resp  

# Parse the gpt JSON response
def custom_json_loader(output_text):
    try:
        analysis = json.loads(output_text)
    except json.JSONDecodeError:
        start = output_text.find("{")
        end = output_text.rfind("}")
        if start != -1 and end != -1 and end > start:
            analysis = json.loads(output_text[start:end+1])
        else:
            raise  
    return analysis

# Service Connector Name
connector = "z-ppp-pr-dbr-keyvaultcredentials-key05"
vault_url = "https://z-ppp-en1-pr-dala-key05.vault.azure.net/"
client_id_key = "modelgateway-healthsgpt-client-id"
client_secret_key = "modelgateway-healthsgpt-client-secret"

# OAuth tenant and scope
tenant_id = "edd791b6-a6e2-450b-9582-5c29c2cc2d25"
scopes = ["https://axapppuk.onmicrosoft.com/modelgateway-api-pr/.default"]

# Model Gateway Base URL
modelgateway_baseurl = "https://proxy.z-ppp-pr-apim01.xpzcloud.com/modelgateway-api-pr/api/"

# Retrieve Secrets via service connector
secrets = get_uc_secrets_map(connector, vault_url, *(client_id_key, client_secret_key))
client_id = secrets[client_id_key]
client_secret = secrets[client_secret_key]

# Mint token via Azure EntraId
token = get_oauth_token(tenant_id, client_id, client_secret, scopes)
print(token)

# Get mol + phone data for combined model output

In [0]:
df_sp = spark.table("axahealth_dataplatform_pd_lab.axahealth_data_scientist_analytics.journeylenz_poc_combinedchannels_fulldata")
# df_sp = spark.table("axahealth_dataplatform_pd_lab.tallulah_coyte_axahealth.validation_priority_review_jl_25826")

In [0]:
df_sp = (df_sp.withColumn(
    "ConversationContent",
    F.regexp_replace(
        "ConversationContent",
        "Customer:Your claim",
        "Agent:Your claim"
    ))
    .withColumn(
        "ConversationContent",
        F.regexp_replace(
            "ConversationContent",
            "Customer: Using our",
            "Agent: Using our"
        )
    )
)

In [0]:
def remove_full_line(transcript: str, strings_to_remove: List[str]) -> str:

    # Split transcript into individual lines
    transcript_lines = transcript.splitlines()

    # Lists to track filtered and removed lines
    filtered_lines = []
    removed_lines = []

    # Create a regex pattern that matches any of the strings to remove
    pattern = re.compile("|".join(re.escape(s) for s in strings_to_remove))

    # First pass: filter out single-word lines that match strings_to_remove
    for i, line in enumerate(transcript_lines):

        if ":" not in line:
            filtered_lines.append(line)
            continue

        speaker, speech = line.split(":", 1)
        word_count = len(speech.split())
        is_single_word = word_count == 1
        is_match = bool(pattern.search(speech))

        if is_single_word and is_match:
            removed_lines.append(line)
        else:
            filtered_lines.append(line)

    # Second pass: merge consecutive lines from the same speaker
    merged_lines = []
    for i, line in enumerate(filtered_lines):

        if ":" not in line:
            merged_lines.append(line)
            continue

        speaker, speech = line.split(":", 1)
        speech = speech.strip()

        if merged_lines and merged_lines[-1].startswith(speaker + ":"):
            merged_lines[-1] += " " + speech
        else:
            merged_lines.append(f"{speaker}: {speech}")

    clean_transcript = "\n".join(merged_lines)

    return clean_transcript


# Create udf of function so it can be used on a pyspark dataframe
remove_lines_udf = F.udf(
    lambda x: remove_full_line(x, ["yeah", "mhm", "ok", "mhm yeah", "sorry", "on"]), StringType()
)


@F.udf(StringType())
def clean_conversation_content(x):

    if x is None:
        print("Input is None, returning None")
        return None

    s = html.unescape(x)

    s = re.sub(r'\|+', '\n', s)
    s = re.sub(r'(?i)<\s*br\s*/?\s*>', '\n', s)
    s = re.sub(r'(?i)</\s*(p|div|li|h[1-6]ul|ol)\s*>', '\n', s)
    s = re.sub(r'<[^>]+>', ' ', s)

    before_speaker_sub = s
    s = re.sub(
        r'\s*(MEMBER|AGENT|ADVISOR|ASSISTANT|ADVISER)\s*:\s*'
        r'(\d{4}-\d{2}-\d{2}\s+\d{2}:d{2}:\d{2})\s*',
        lambda m: f"\n{m.group(1).upper()}: {m.group(2)}\n",
        s,
        flags=re.IGNORECASE
    )

    s = re.sub(r'[ \t]+', ' ', s)
    s = re.sub(r' *\n *', '\n', s)
    s = re.sub(r'\n{3,}', '\n\n', s)

    result = s.strip()

    return result


In [0]:
df_sp = (
    df_sp
    .withColumn(
        "CleanConversationContent",
        F.when(
            F.col("ContactChannel") == "MOL",
            clean_conversation_content(F.col("ConversationContent"))
        )
        .when(
            F.col("ContactChannel") == "Telephone",
            remove_lines_udf(F.col("ConversationContent"))
        ).otherwise(F.col("ConversationContent"))
    )
)

In [0]:
df_sp = df_sp.filter(F.col("MSKClaim") == 1)
df_sp = df_sp.filter(F.col("ContactChannel") != "LiveChat") # remove when LiveChat transcripts are live

In [0]:
# toPandas() method for a large pyspark dataframe
# Define 3 date range chunks
chunks = [
    # ("2025-07-01", "2025-11-30"),  # Chunk 1: Jul–Nov 2025 UNCOMMENTED ON 30/8/26 16:00
    # ("2025-12-01", "2026-01-09"),  # Chunk 2: Dec 2025–jan 2026 UNCOMMENT ON NEXT RUN 28/8/26 10:37, RAN ON 30/8/26,
    ("2026-01-10", "2026-03-01"), # chunk 2.25: jan-feb26 RAN ON 1/9/26 9:33
    # ("2026-03-02", "2026-04-30"), # CHUNK 2.5 (DATA TOO BIG): FEB-APR 2026 UNCOMMENT ON NEXT RUN 30/8/26 19:18
    # ("2026-05-01", "2026-08-26")   # Chunk 3: May–Aug 2026 DO NOT UNCOMMENT ON NEXT RUN 28/8/26 10:37
]

pandas_dfs = []
for start, end in chunks:
    df_chunk = df_sp.filter(
        (F.col("ConversationStartTime") >= start) &
        (F.col("ConversationStartTime") <= end)
    )
    pandas_dfs.append(df_chunk.toPandas())

df = pd.concat(pandas_dfs, ignore_index=True)

# # toPandas() method for a small/medium pyspark dataframe
# df = df_sp.toPandas()

df.shape

In [0]:
df.head()

# Set-up inference + prompt

In [0]:
# =========================
# CONFIG
# =========================

import asyncio
import json
import time
import math
import random
import traceback
import pandas as pd
import numpy as np
from rapidfuzz.fuzz import partial_ratio
from collections import deque

MODEL = "gpt-4o-2024-11-20"
MAX_CONCURRENCY = 20
MIN_TRANSCRIPT_CHARS = 40
MAX_TOKENS = 800
RETRIES = 3

ENABLE_CONSUMPTION_ENRICHMENT = False
CONSUMPTION_MAX_RETRIES = 6
CONSUMPTION_BASE_WAIT = 1.0

# =========================
# JSON SCHEMA
# =========================
BUCKETS = {
    "Start a claim": {
        "SC1 – Report a new health issue for a claim":
            "Customer describes symptoms/injury/diagnosis and wants AXA to open or authorise a claim (e.g. get a claim number, book first appointment).",
        "SC2 – Check if a health issue is claimable or covered before starting":
            "Customer asks if a symptom/condition/treatment would be covered or claimable, but is mainly seeking eligibility info rather than starting the claim."
    },

    "Progress my claim": {
        "PC1 – Check claim progress or outcome":
            "Customer asks about status or result of an existing claim (received, authorised, paid, decision made).",
        "PC2 – Provide or correct claim-related information":
            "Customer supplies or amends information/documents that AXA needs to process a claim (referrals, letters, contact/bank details, etc.).",
        "PC3 – Understand or resolve delays":
            "Customer asks why a claim is taking time or when it will be processed; focuses on timing/waiting.",
        "PC4 – Understand decisions on rejected or partially paid claims":
            "Customer asks why a claim/treatment was rejected, only partially covered, or appears incorrectly paid."
    },

    "Help with confusion": {
        "HC1 – Clarify limits, exclusions or financial responsibilities":
            "Customer wants to understand excesses, limits, guided options, shortfalls, or who pays what under the policy.",
        "HC2 – Understand or resolve billing/invoice discrepancies":
            "Customer queries specific invoices, bills, duplicate charges, or differences between provider bills and AXA statements.",
        "HC3 – Get help with online or account access issues":
            "Customer has trouble with the online portal/app (logging in, uploading documents, viewing messages or claims)."
    },

    "Clarification": {
        "CL1 – Understand claim or treatment pathway (step-by-step)":
            "Customer asks for an end‑to‑end explanation of the process (GP → referral → authorisation → treatment → billing, etc.).",
        "CL2 – Understand specific pre-authorisation or referral requirements":
            "Customer asks targeted questions about referral/pre‑authorisation rules (e.g. need GP first, need written referral, who can refer)."
    },

    "Understand policy benefits or entitlement": {
        "BE1 – Understand overall coverage and benefit limits":
            "Customer asks generally what their policy covers, what limits/excesses exist, or what benefits they have (not tied to a specific booked event).",
        "BE2 – Check cover for a planned procedure or course of treatment":
            "Customer asks if a specific upcoming procedure/treatment/hospital/doctor will be covered.",
        "BE3 – Understand coverage or benefits for dependants":
            "Customer asks about coverage, limits or entitlements for dependants (spouse/children) on the policy."
    }
}
 

JSON_SCHEMA = {
    "name": "call_analysis",
    "schema": {
        "type": "object",
        "properties": {
            "topics": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "topic": {"type": "string"},
                        "time_of_contact": {
                            "type": ["string", "null"],
                            "description": "Timestamp for the bucket. Use YYYY-MM-DDTHH:MM:SS. For telephone use the call timestamp; for MOL use the timestamp of the message that supports the topic."
                        },
                        "bucket": {
                            "type": "string",
                            "enum": list(BUCKETS.keys()),
                            "description": "Top-level taxonomy category. Must match exactly one of the defined buckets."
                        },
                        "sub_bucket": {
                            "type": "string",
                            "enum": [sb for subs in BUCKETS.values() for sb in subs],
                            "description": (
                                "Sub-topic code. Must belong to the selected bucket. "
                                "Valid combinations:\n" +
                                "\n".join(
                                    f"  {bucket}: {', '.join(subs.keys())}"
                                    for bucket, subs in BUCKETS.items()
                                )
                            )
                        },
                        "coverage_pct": {
                            "type": "number",
                            "minimum": 0,
                            "maximum": 100
                        },
                        "sentiment": {
                            "type": "integer",
                            "minimum": -5,
                            "maximum": 5
                        },
                        "resolved": {
                            "type": "boolean"
                        },
                        "reasons_for_contact": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "properties": {
                                    "description": {"type": "string"},
                                    "evidence": {"type": "string"},
                                    "suggestion": {"type": "string"},
                                },
                                "required": ["description", "evidence", "suggestion"],
                                "additionalProperties": False
                            },
                            "minItems": 1
                        }
                    },
                    "required": [
                        "time_of_contact",
                        "topic",
                        "bucket",
                        "sub_bucket",
                        "coverage_pct",
                        "sentiment",
                        "resolved",
                        "reasons_for_contact",
                    ],
                    "additionalProperties": False
                },
                "minItems": 1
            },
            "overall_resolution": {"type": "boolean"}
        },
        "required": ["topics", "overall_resolution"],
        "additionalProperties": False
    }
}

ALLOWED_BUCKETS = set(
    JSON_SCHEMA["schema"]["properties"]["topics"]["items"]["properties"]["bucket"]["enum"]
)


# =========================
# PROMPT
# =========================

def build_prompt(transcript):
    def build_taxonomy_text(buckets: dict) -> str:
        lines = []
        for bucket, sub_buckets in buckets.items():
            lines.append(f"\n**{bucket}**")
            for sub_code, definition in sub_buckets.items():
                lines.append(f"  - {sub_code}: {definition}")
        return "\n".join(lines)

    taxonomy_text = build_taxonomy_text(BUCKETS)

    return f"""
    You are an expert data extraction assistant at AXA Health. Your task is to analyse a customer transcript and extract structured demand topics according to the rules below.

    ---

    ## INPUT

    **Transcript:**
    {transcript}

    ---

    ## OUTPUT REQUIREMENTS

    - Return ONLY a valid JSON object that conforms exactly to the schema provided below.
    - Do NOT include markdown, code fences, commentary, or any text outside the JSON object.

    ---

    ## CLASSIFICATION RULES

    ### Taxonomy & Bucketing
    - Every topic MUST be assigned a `bucket` and a `sub_bucket`.
    - `bucket` corresponds to a top-level taxonomy category (e.g. "Start a claim", "Progress my claim").
    - `sub_bucket` MUST be one of the defined sub-topics belonging to its parent `bucket`.
    - Use the sub-topic **definitions** provided in the taxonomy to guide your selection — 
    choose the sub_bucket whose definition best matches the customer's intent in the transcript.
    - Invalid bucket/sub_bucket combinations are strictly forbidden.

    ### Sub-Bucket Selection Guidance
    Use the definitions below to distinguish between similar sub-topics:

    {taxonomy_text}

    ### Topics
    - Return topics as a list; return reasons_for_contact as a list.
    - You MUST include at least 1 topic.
    - Use multiple topics ONLY when the transcript genuinely covers distinct customer intents.
    - Topic name: maximum 3 words.

    ---

    ## FIELD-LEVEL RULES

    ### Sentiment (`sentiment_score`)
    - Assign sentiment at the bucket/sub_bucket level (one score per bucket).
    - Scale: -5 (extremely negative) to 0 (neutral) to +5 (extremely positive).
    - Sentiment reflects the customer's attitude toward the company, service quality, 
    claims experience, communication, delays, or admin issues.
    - Negative sentiment applies ONLY when the customer expresses frustration, 
    dissatisfaction, confusion, complaints, poor service, delays, or admin problems 
    attributable to the company or claims process.
    - Do NOT infer positivity from politeness alone.
    - Do NOT assign negative sentiment solely because the customer is injured, ill, 
    or discussing a medical condition.

    ### Resolution (`resolved` and `overall_resolution`)
    - Assign `resolved` at the bucket/sub_bucket level (one boolean per bucket).
    - Set `overall_resolution = true` ONLY if the customer's overall issue appears resolved 
    by the end of the transcript.

    ### Time of Contact (`time_of_contact`)
    - Populate for every bucket returned.
    - **Telephone calls:** Use the call start timestamp shown at the beginning of the 
    transcript. Apply the same timestamp to all topics.
    - **Member online messages:** Use the timestamp of the message in which each topic 
    is first introduced. If a topic spans multiple messages, use the earliest 
    relevant timestamp.
    - Format: ISO 8601 (`YYYY-MM-DDTHH:MM:SS`). If no timestamp can be identified, 
    return `null`.

    ### Evidence / Reasons
    - Include at least 1 reason per topic.
    - Reasons are supporting evidence only — not summaries.
    - Each evidence string must be ≤ 140 characters.
    - Prefer verbatim quotes; paraphrase only when verbatim is not possible.
    - Redact any PII within evidence as `[REDACTED]`.

    ### Coverage (`coverage_pct`)
    - Estimate the percentage of the transcript spent on each topic.
    - Value must be between 0 and 100.
    - The sum of all `coverage_pct` values across all topics should be approximately 100.
    - Small/incidental mentions → low %; dominant topics → high %.

    ### Clarification Flag
    - Use `clarification = true` ONLY when:
    - The customer explicitly expresses confusion, OR
    - The customer asks for something to be repeated or re-explained.
    - Do NOT flag as clarification for general questions about claims, policy coverage, 
    payments, or membership.
    - NOTE: The "Clarification" bucket (CL1, CL2) is a taxonomy category about *process 
    understanding*, and is independent of the `clarification` boolean field above

    {JSON_SCHEMA}
    """.strip()

# Helpers

In [0]:
#================
# PII Redaction
#=================
PII_PATTERNS = [
    re.compile(r"\b\d{10,}\b"),                        # long digit sequences (e.g., policy numbers)
    re.compile(r"\b(?:\+?\d[\d\-\s]{6,}\d)\b"),        # phone numbers
    re.compile(r"[A-Za-z0-9\.\-_]+@[A-Za-z0-9\.\-]+\.[A-Za-z]{2,}"),  # emails
    re.compile(r"\b(?:DOB|D\.O\.B\.|Date of Birth)\b.*", re.IGNORECASE),  # DOB phrases (rough)
    re.compile(r"\b[A-Z]{2}\d{6,}\b"),                 # generic alphanumeric IDs like AB123456
    re.compile(r"\b(?:membership|policy|claim)\s*(?:no\.?|number|id)?\s*[:#\-]?\s*\w+\b", re.IGNORECASE)
]

def redact_pii(text: str) -> str:
    if not isinstance(text, str):
        return ""
    redacted = text
    for pat in PII_PATTERNS:
        redacted = pat.sub("[REDACTED]", redacted)
    return redacted.strip()

#============
#JOSN Parsing / Repair
#=============
def parse_model_content(content: str) -> Dict[str, Any]:
    """
    Try strict JSON first, then fallback to extracting the outermost JSON object
    """
    try:
        return json.loads(content)
    except json.JSONDecodeError:
        start = content.find("{")
        end = content.find("}")
        if start != -1 and end != -1 and end > start:
            return json.loads(content[start:end + 1])
        raise

def clamp_sentiment(x: Any) -> int:
    try:
        val = int(x)
    except Exception:
        val = 0
    return max(-5, min(5, val))

def normalise_bucket(bucket: Any) -> str:
    bucket = str(bucket or "").strip()
    if bucket in ALLOWED_BUCKETS:
        return bucket
    #fallback default bucket
    return "Clarification"

def normalise_coverage(topics):
    if not topics:
        return topics
    
    total = sum(t.get("coverage_pct", 0) for t in topics)

    if total == 0:
        #fallback - equal distribution
        n = len(topics)
        for t in topics:
            t["coverage_pct"] = 100 / n
        return topics
    
    for t in topics:
        t["coverage_pct"] = (t.get("coverage_pct", 0) / total) * 100

    return topics

def validate_and_repair(analysis: Any) -> Dict[str, Any]:
    """
    Makes output safe for downstream use even if the model is imperfect.
    """
    if not isinstance(analysis, dict):
        analysis = {}

    topics = analysis.get("topics", [])
    if not isinstance(topics, list):
        topics = []

    cleaned_topics: List[Dict[str, Any]] = []
    any_unresolved = False

    for t in topics:
        if not isinstance(t, dict):
            continue

        topic_name = str(t.get("topic", "")).strip()
        if not topic_name:
            topic_name = "Unknown topic"
        
        bucket = normalise_bucket(t.get("bucket"))

        sub_bucket = str(t.get("sub_bucket", "")).strip()
        if not sub_bucket:
            sub_bucket = "Other"

        reasons = t.get("reasons_for_contact", [])
        if not isinstance(reasons, list):
            reasons = []

        cleaned_reasons: List[Dict[str, Any]] = []

        for r in reasons:
            if not isinstance(r, dict):
                continue

            description = str(r.get("description", "")).strip()
            evidence = redact_pii(r.get("evidence", ""))[:140]
            resolved = bool(r.get("resolved", False))
            sentiment = clamp_sentiment(r.get("sentiment", 0))
            suggestion = str(r.get("suggestion", "")).strip()

            if not description:
                continue

            if not resolved:
                any_unresolved = True

            cleaned_reasons.append({
                "description": description,
                "evidence": evidence,
                "resolved": resolved,
                "sentiment": sentiment,
                "suggestion": suggestion
            })

        if cleaned_reasons:
            cleaned_topics.append({
                "topic": topic_name[:60],
                "bucket": bucket,
                "sub_bucket": sub_bucket,
                "coverage_pct": t.get("coverage_pct", 0),
                "reasons_for_contact": cleaned_reasons
            })

    cleaned_topics = normalise_coverage(cleaned_topics)
            
    overall_resolution = analysis.get("overall_resolution", None)
    if not isinstance(overall_resolution, bool):
        overall_resolution = not any_unresolved if cleaned_topics else False

    return {
        "topics": cleaned_topics,
        "overall_resolution": overall_resolution
    }

# Model Call + Process return
## Handles:
- Calling model
- extracts response
- parses JSON
- validates + repairs using helpers
- extracts token usage stats
- records API latency
- captures request ID for traceability
- Retry logic
- Waits progressivel longer between retires (exponential backoff)
- Only throws exception after max retry count has been reached

In [0]:
# =========================
# MODEL CALL
# =========================

def call_model(prompt, max_tokens: int = MAX_TOKENS, retries: int = RETRIES) -> Tuple[Dict[str, Any], Dict[str, Any]]:
    apiurl = f"{modelgateway_baseurl}/secure-gpt-openai/openai/deployments/{MODEL}/chat/completions?api-version=2024-06-01"

    payload = {
        "messages": [
            {"role": "system", "content": "You analyse customer service transcripts."},
            {"role": "user", "content": prompt}
        ],
        "temperature": 0,
        "max_tokens": max_tokens,
        "response_format": {
            "type": "json_schema",
            "json_schema": {
                "name": JSON_SCHEMA["name"],
                "schema": JSON_SCHEMA["schema"],
                "strict": True
            }
        }
    }

    last_exception = None

    for attempt in range(retries):
        start_api = time.time()
        try:
            resp = post_with_bearer(apiurl, token, payload)
            request_id = None
            try:
                request_id = resp.headers.get("x-request-id")
            except Exception:
                pass

            api_latency = time.time() - start_api

            r = resp.json()
            choice = r["choices"][0]["message"]
            if "parsed" in choice:
                parsed = choice["parsed"]
            elif choice.get("content"):
                parsed = json.loads(choice["content"])
            else:
                raise ValueError("No parsable content returned from model")

            cleaned = validate_and_repair(parsed)

            usage = r.get("usage")
            if not usage:
                raise ValueError(f"No usage returned: {r}")

            prompt_tokens = usage.get("pompt_tokens")
            completion_tokens = usage.get("completion_tokens")
            total_tokens = usage.get("total_tokens")

            if prompt_tokens is None:
                if total_tokens is not None and completion_tokens is not None:
                    try:
                        prompt_tokens = int(total_tokens) - int(completion_tokens)
                    except Exception:
                        prompt_tokens = None

            meta = {
                "prompt_tokens": prompt_tokens,
                "completion_tokens": completion_tokens,
                "total_tokens": total_tokens,
                "api_latency": api_latency,
                "request_id": request_id,
                "attempt": attempt + 1
            }

            return parsed, meta

        except Exception as e:
            if attempt == retries - 1:
                raise e
            time.sleep(2 ** attempt)

    raise RuntimeError(f"Model call failed after {retries} attempts: {last_exception}")

# =========================
# ANALYSE ONE
# =========================

async def analyse_one(transcript: Any):
    transcript = str(transcript or "").strip()

    if len(transcript) < MIN_TRANSCRIPT_CHARS:
        return {"topics": [], "overall_resolution": False}, None, None, None, 0.0, 0.0, None, None

    try:
        start_task = time.time()
        result, meta = await asyncio.to_thread(
            call_model,
            build_prompt(transcript)
        )
        task_latency = time.time() - start_task

        prompt_tokens = meta.get("prompt_tokens")
        completion_tokens = meta.get("completion_tokens")
        total_tokens = meta.get("total_tokens")
        api_latency = meta.get("api_latency", 0.0)
        rid = meta.get("request_id", None)

        if prompt_tokens is None and total_tokens and completion_tokens is not None:
                prompt_tokens = total_tokens - completion_tokens

        return result, prompt_tokens, completion_tokens, total_tokens, api_latency, task_latency, rid, None

    except Exception as e:
        raise e

# =======================
# FLATTEN
# =======================

def flatten(analysis, row_id):
    rows = []
    for t in analysis.get("topics", []):
        for r in t.get("reasons_for_contact", []):
            rows.append({
                "row_id": row_id,
                "time_of_contact": t.get("time_of_contact"),
                "topic": t.get("topic"),
                "bucket": t.get("bucket"),
                "sub_bucket": t.get("sub_bucket"),
                "coverage_pct": t.get("coverage_pct"),
                "description": r.get("description"),
                "evidence": r.get("evidence"),
                "evidence_score": r.get("evidence_score"),
                "suggestion": r.get("suggestion"),
                "sentiment": r.get("sentiment"),
                "resolved": r.get("resolved"),
                "overall_resolution": analysis.get("overall_resolution")
            })

    return rows

#==============
# SUMMARY METRICS PER TRANSCRIPT
#===========
def summarise_analysis(analysis: Any) -> Dict[str, Any]:
    if not isinstance(analysis, dict):
        return {
            "num_topics": 0,
            "num_reasons": 0,
            "avg_sentiment": None,
            "overall_resolution": None
        }

    topics = analysis.get("topics", [])
    num_topics = len(topics)
    num_reasons = 0
    sentiments: List[int] = []

    for t in topics:
        reasons = t.get("reasons_for_contact", [])
        num_reasons += len(reasons)
        for r in reasons:
            try:
                sentiments.append(int(r.get("sentiment", 0)))
            except Exception:
                pass
    avg_sentiment = float(np.mean(sentiments)) if sentiments else None

    return {
        "num_topics": num_topics,
        "num_reasons": num_reasons,
        "avg_sentiment": avg_sentiment,
        "overall_resolution": analysis.get("overall_resolution", False)
    }

#================
#HELPERS
#===============
def build_nested(g, total_talk_secs, contact_channel):
    if g.empty or "bucket" not in g.columns:
        return []
    
    buckets = []
    for (bucket_name, sub_bucket_name), bg in g.groupby(["bucket", "sub_bucket"], dropna=False):
        total_coverage = g["coverage_pct"].sum()
        if total_coverage == 0:
            coverage = 0
        else:
            coverage = float(bg["coverage_pct"].sum() / total_coverage * 100)

        if contact_channel == "MOL":
            talk_time_sec = 0
            talk_time_mins = 0
        else:
            talk_time_sec = (coverage / 100.0) * total_talk_secs
            talk_time_mins = round(talk_time_sec / 60.0, 2)

        bucket_sentiment = (
            round(float(bg["sentiment"].dropna().mean()), 2)
            if bg["resolved"].notna().any()
            else None
        )

        bucket_resolved = (
            bool(bg["resolved"].all())
            if bg["resolved"].notna().any()
            else None
        )

        bucket_time = (
            bg["time_of_contact"].dropna().iloc[0]
            if "time_of_contact" in bg.columns and bg["time_of_contact"].notna().any()
            else None
        )

        bucket_entry = {
            "time_of_contact": bucket_time,
            "bucket": bucket_name,
            "sub_bucket": sub_bucket_name or "Uknown sub_bucket",
            "coverage_pct": coverage,
            "talk_time_secs": round(talk_time_sec, 1),
            "talk_time_mins": talk_time_mins,
            "sentiment": bucket_sentiment,
            "resolved": bucket_resolved,
            "topics": []
        }

        for topic_name, tg in bg.groupby("topic"):

            topic_entry = {
                "topic": topic_name,
                "reasons": [],
            }

            for _, r in tg.iterrows():
                topic_entry["reasons"].append({
                "description": r["description"],
                "evidence": r["evidence"],
                "evidence_score": r.get("evidence_score"),
                "suggestion": r.get("suggestion"),
            })
            
            bucket_entry["topics"].append(topic_entry)
        
        buckets.append(bucket_entry)
    
    return buckets

def build_nested_from_analysis(analysis, total_talk_secs, transcript, contact_channel):

    transcript_l = (transcript or "").lower()

    #print("Transcript length:", len(transcript_l))

    if not analysis or not isinstance(analysis, dict):
        return []
    flat_rows = []

    for topic in analysis.get("topics", []):
        time_of_contact = topic.get("time_of_contact")
        bucket = topic.get("bucket")
        sub_bucket = topic.get("sub_bucket")
        topic_name = topic.get("topic")
        coverage_pct = float(topic.get("coverage_pct", 0) or 0)
        sentiment = topic.get("sentiment")
        resolved = topic.get("resolved")

        for reason in topic.get("reasons_for_contact", []):

            evidence_score = score_reason_evidence(reason, transcript_l)

            flat_rows.append({
                "time_of_contact": time_of_contact,
                "bucket": bucket,
                "sub_bucket": topic.get("sub_bucket"),
                "topic": topic_name,
                "coverage_pct": coverage_pct,
                "sentiment": sentiment,
                "resolved": resolved,
                "description": reason.get("description"),
                "evidence": reason.get("evidence"),
                "evidence_score": evidence_score,
                "suggestion": reason.get("suggestion"),
            })

    if not flat_rows:
        return []
    g = pd.DataFrame(flat_rows)
    if "sub_bucket" not in g.columns:
        g["sub_bucket"] = "Unknown sub_bucket"
    return build_nested(g, total_talk_secs, contact_channel)

def weighted_sentiment(buckets):
    total = 0
    weight_sum = 0

    for b in buckets:
        weight = b.get("coverage_pct", 0)
        sentiment = b.get("sentiment")

        if sentiment is None:
            continue

        total += sentiment * weight
        weight_sum += weight

    return total / weight_sum if weight_sum else None

def get_first_bucket(buckets):
    if not buckets:
        return None
    b = buckets[0]
    bucket = b.get("bucket")
    sub_bucket = b.get("sub_bucket")
    return f"{bucket} -> {sub_bucket}" if sub_bucket else bucket

def get_dominant_bucket(buckets):
    if not buckets:
        return None
    b = max(
        buckets,
        key=lambda t: t.get("talk_time_secs", 0)
    )
    bucket = b.get("bucket")
    sub_bucket = b.get("sub_bucket")
    return f"{bucket} -> {sub_bucket}" if sub_bucket else bucket

def best_match_score(evidence, transcript):
    if not evidence or not transcript:
        return 0.0
    
    return partial_ratio(evidence, transcript) / 100.0

def score_reason_evidence(reason, transcript_l):
    evidence = clean_text(reason.get("evidence") or "")
    description = clean_text(reason.get("description") or "")

    #print("=======DEBUG========")
    #print("Evidence:", repr(evidence))
    #print("Transcript:", repr(transcript_l[:200] if transcript_l else None))

    if not evidence or not transcript_l:
        return 0.0
    
    match_score = best_match_score(evidence, transcript_l)

    desc_words = set(description.split())
    ev_words = set(evidence.lower().split())

    overlap = 0
    if desc_words:
        overlap = len(desc_words & ev_words) / len(desc_words)

    final_score = (0.85 * match_score) + (0.15 * overlap)

    return round(min(final_score, 1.0), 3)

def clean_text(s):
    return re.sub(r"[^\w\s]", "", s.lower())

def estimate_tokens(text: str) -> int:
    if not isinstance(text, str):
        return 1500
    est = len(text) // 4
    return max(500, min(est, 8000))

async def refresh_token_async():
    global token
    loop = asyncio.get_event_loop()
    token = await loop.run_in_executor(
        None,
        lambda: get_oauth_token(tenant_id, client_id, client_secret, scopes)
    )
    print("Token Refreshed")

# Run Batch

In [0]:
# =======================
# RUN BATCH
# =========================
MAX_RETRIES = 5 #number of retries on fail
RATE_LIMIT_WAIT = 60 * 5 #5mins
BATCH_SIZE = 20 #for each thread
TOKEN_LIMIT = 15_000_000 #token limit for 30 min window
WINDOW_SECONDS = 60 * 30 #30mins
SAFETY_BUFFER = 0.95 #buffer relates to max tokens available in window
token_usage_window = deque()
current_tokens = 0
token_lock = asyncio.Lock()
CHECKPOINT_TABLE = "axahealth_dataplatform_pd_lab.axahealth_data_scientist_analytics.journeylenz_output_checkpoint_full_data_run_2_1_3"
output_schema = StructType([
    StructField("ConversationContent", StringType(), True),
    StructField("MembershipNumber", StringType(), True),
    StructField("ClaimNumber", StringType(), True),
    StructField("ConversationId", StringType(), True),
    StructField("ConversationTimeSecs", DoubleType(), True),
    StructField("analysis", StringType(), True),
    StructField("prompt_tokens", IntegerType(), True),
    StructField("completion_tokens", IntegerType(), True),
    StructField("total_tokens", IntegerType(), True),
    StructField("api_latency", DoubleType(), True),
    StructField("task_latency", DoubleType(), True),
    StructField("request_id", StringType(), True),
    StructField("error", StringType(), True),
])

async def clean_token_window():
    global current_tokens
    now = time.time()

    async with token_lock:
        while token_usage_window and now - token_usage_window[0][0] > WINDOW_SECONDS:
            _, old_tokens = token_usage_window.popleft()
            current_tokens = max(0, current_tokens - old_tokens)

async def throttle_if_needed(estimated_tokens=1500):
    global current_tokens

    await clean_token_window()
    limit = TOKEN_LIMIT * SAFETY_BUFFER
    while current_tokens + estimated_tokens > limit:
        pressure = current_tokens / limit
        sleep_time = min(1 + pressure * 4, 8)
        print(f"[THROTTLE] Tokens {current_tokens}/{TOKEN_LIMIT}")
        await asyncio.sleep(sleep_time)
        await clean_token_window()

async def run(df_in: pd.DataFrame):
    required_cols = [
        "ConversationContent",
        "ConversationId",
    ]
    missing = [c for c in required_cols if c not in df_in.columns]
    if missing:
        raise ValueError(f"Input dataframe is missing required columns: {missing}.")
    df = df_in.reset_index(drop=True).copy()
    #convert to spark
    input_sdf = spark.createDataFrame(df_in)
    #make sure the comparison columns have compatible types
    input_sdf = (
        input_sdf
        .withColumn("ConversationId", F.col("ConversationId").cast("string"))
    )
    #remove records already present in checkpoint
    if spark.catalog.tableExists(CHECKPOINT_TABLE):
        checkpoint_keys = (
            spark.table(CHECKPOINT_TABLE)
            .select(
                F.col("ConversationId").cast("string").alias("ConversationId")
                )
            .where(
                F.col("ConversationId").isNotNull()
            )
            .dropDuplicates(
                ["ConversationId"]
            )
        )
        before = input_sdf.count()

        input_sdf = input_sdf.join(
            checkpoint_keys,
            on="ConversationId",
            how="left_anti",
        )

        after = input_sdf.count()
        print(f"[RESUME] Skipped {before - after} already processed rows")
        print(f"[RESUME] Rows remaining: {after}")
    else:
        print("[RESUME] No checkpoint found, starting fresh")
    #nothing new to process
    if input_sdf.limit(1).count() == 0:
        print(f"[RESUME] Nothing new to process.")
        return None
    #convert only the remaining rows back to pandas
    df = input_sdf.toPandas().reset_index(drop=True)
    print(f"Using MAX_CONCURRENCY={MAX_CONCURRENCY}, BATCH_SIZE={BATCH_SIZE}")

    global TOTAL_RECORDS, processed, start_time
    TOTAL_RECORDS = len(df)
    processed = 0
    start_time = time.time()
    semaphore = asyncio.Semaphore(MAX_CONCURRENCY)
    rate_limit_event = asyncio.Event()
    rate_limit_event.set()
    rate_limit_lock = asyncio.Lock()
    is_rate_limited = False
    is_refreshing_token = False

    async def task(t, membership, claim, conversation):
        global current_tokens, processed
        nonlocal is_rate_limited, is_refreshing_token
        retries = 0
        while retries < MAX_RETRIES:
            await rate_limit_event.wait()
            try:
                async with semaphore:
                    await asyncio.sleep(0.05 + random.uniform(0, 0.01)) #soft jitter
                    estimated_tokens = estimate_tokens(t)
                    await throttle_if_needed(estimated_tokens)
                    result = await analyse_one(t)
                    analysis, prompt_tokens, completion_tokens, row_total_tokens, api_latency, task_latency, rid, err = result
                    tokens_used = row_total_tokens or 0
                    async with token_lock:
                        token_usage_window.append((time.time(), tokens_used))
                        current_tokens += tokens_used
                        processed += 1

                    if processed % MAX_CONCURRENCY == 0 or processed == TOTAL_RECORDS:
                        elapsed = time.time() - start_time
                        rate = processed / elapsed if elapsed > 0 else 0
                        pct = (processed / TOTAL_RECORDS) * 100
                        remaining = TOTAL_RECORDS - processed
                        eta = remaining / rate if rate > 0 else 0
                        avg_latency = elapsed / processed if processed > 0 else 0
                        print(
                            f"""
                            [PROGRESS]
                            Processed: {processed}/{TOTAL_RECORDS} ({pct:.2f}%)
                            Speed: {rate:.2f} row/sec
                            Avg time/row: {avg_latency:.2f}s
                            Elapsed: {elapsed:.1f}s
                            ETA: {eta:.1f}s
                            Tokens: {current_tokens}/{TOKEN_LIMIT}
                            """
                        )

                    if err:
                        raise Exception(err)
                    return (
                        analysis,
                        prompt_tokens,
                        completion_tokens,
                        row_total_tokens,
                        api_latency,
                        task_latency,
                        rid,
                        err
                    )
                
            except Exception as e:
                err_str = str(e)
                print(err_str)
                #print("retry loop hit:", err_str)
                if "429" in err_str or "rate_limit" in err_str.lower():
                    wait_time = RATE_LIMIT_WAIT
                    async with rate_limit_lock:
                        if not is_rate_limited:
                            is_rate_limited = True
                            rate_limit_event.clear()
                            print(f"Pausing ALL tasks for {wait_time/60:.1f} mins due to rate limit...")
                            for attempt in range(MAX_RETRIES):
                                sleep_time = min(15 * (2 ** attempt), 60)
                                print(f"[RATE LIMIT] Backing off {sleep_time}s (attempt {attempt+1})...")
                                await asyncio.sleep(sleep_time)
                                await clean_token_window()

                                if current_tokens < TOKEN_LIMIT * SAFETY_BUFFER * 0.9:
                                    print("[RATE LIMIT] Token pressure reduced -> resuming")
                                    break

                            print("Resuming after rate limit reset")
                            rate_limit_event.set()
                            is_rate_limited = False
                    retries += 1
                    continue
                elif ("401" in err_str
                    or "403" in err_str
                    or "unathorized" in err_str.lower()
                    or "token" in err_str.lower()
                    ):
                    async with token_lock:
                        if not is_refreshing_token:
                            is_refreshing_token = True
                            print("Token expired (row {correlation}) -> refreshing...")
                            await refresh_token_async()
                            print("Token refresh complete")
                            is_refreshing_token = False
                        else:
                            print("Waiting for token refresh (row {correlation})...")
                            await asyncio.sleep(1)
                    retries += 1
                    continue

                #Other errors-> fail fast
                return None, None, None, None, 0.0, 0.0, None, str(e)
        return None, None, None, None, 0.0, 0.0, None, "Max retries exceeded"

    #results = []
    items = list(zip(
        df["ConversationContent"], 
        df["MembershipNumber"],
        df["ClaimNumber"],
        df["ConversationId"],
        df["ConversationTimeSecs"]))
    for i in range(0, len(items), BATCH_SIZE):
        batch = items[i:i+BATCH_SIZE]
        batch_num = i // BATCH_SIZE + 1
        total_batches = (len(items) + BATCH_SIZE - 1) // BATCH_SIZE
        print(f"[BATCH] {batch_num}/{total_batches} | Processed: {processed}/{TOTAL_RECORDS}")
        batch_results = await asyncio.gather(
            *[task(t, m, c, conv) for t, m, c, conv, talk_time_secs in batch]
        )
        rows = [
            (
                t,
                membership,
                claim,
                conversation,
                talk_time_secs,
                analysis,
                pt,
                ct,
                tt,
                al,
                tl,
                rid,
                err
            )
            for (
                (t, membership, claim, conversation, talk_time_secs),
                (analysis, pt, ct, tt, al, tl, rid, err))
            in zip(batch, batch_results)
        ]
        sdf = spark.createDataFrame(rows, schema=output_schema)
        sdf.write.mode("append").format("delta").saveAsTable(CHECKPOINT_TABLE)
        if batch_num % 20 == 0:
            print(f"[CHECKPOINT] Saved up to batch {batch_num} ({processed} rows)")

    return None

async def timed_run(df):
    start = time.time()
    await run(df)
    sdf = spark.table(CHECKPOINT_TABLE).select(
        "ConversationContent",
        "MembershipNumber",
        "ClaimNumber",
        "ConversationId",
        "ConversationTimeSecs",
        "analysis",
        "prompt_tokens",
        "completion_tokens",
        "total_tokens",
        "api_latency",
        "task_latency",
        "request_id",
        "error"
    )
    df = sdf.toPandas()
    df["analysis"] = df["analysis"].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )

    df["topics_struct"] = df.apply(
        lambda row: build_nested_from_analysis(
            row["analysis"],
            float(row.get("ConversationTimeSecs", 0) or 0),
            str(row.get("ConversationContent") or ""),
            str(row.get("ContactChannel"))
        ),
        axis=1
    )
    df["overall_sentiment"] = df["topics_struct"].apply(weighted_sentiment)
    df["first_bucket"] = df["topics_struct"].apply(get_first_bucket)
    df["dominant_bucket"] = df["topics_struct"].apply(get_dominant_bucket)

    wall_time = time.time() - start

    metrics = {
        "tokens": {
            "prompt": int(df["prompt_tokens"].fillna(0).sum()),
            "completion": int(df["completion_tokens"].sum()),
            "total": int(df["total_tokens"].sum())
        },
        "latency": {
            "avg_task_sec": float(df["task_latency"].mean()),
            "avg_api_sec": float(df["api_latency"].mean()),
        },
        "throughput": {}
    }

    records_per_sec = len(df) / wall_time if wall_time > 0 else 0
    metrics["throughput"]["records_per_sec"] = records_per_sec
    metrics["throughput"]["wall_time_sec"] = wall_time

    print(f"Pipeline runtime (wall time): {wall_time:.2f}s")
    #print(json.dumps(metrics, indent=2))
    return df, metrics, wall_time

# =========================
# FINAL
# =========================
df_out, metrics, wall_time = await timed_run(df)
display(df_out)

# Build dreictly from df_out (in memory). View later cells for rebuilding code from checkpoint

In [0]:
df_out["topics_pretty"] = df_out["topics_struct"].apply(
    lambda x: json.dumps(x, indent=2)
)
display(df_out[["MembershipNumber", "ClaimNumber", "ConversationId", "ConversationContent", "prompt_tokens", "completion_tokens", "total_tokens", "topics_pretty", "overall_sentiment", "first_bucket", "dominant_bucket"]])

In [0]:
df_out_clean = df_out.drop(columns=["analysis"])

In [0]:
object_cols = [
    "error",
    "topics_struct",
    "first_bucket",
    "dominant_bucket",
    "topics_pretty"
]
for c in object_cols:
    df_out_clean[c] = df_out_clean[c].astype(str)

In [0]:
#write initial output
spark_df = spark.createDataFrame(df_out_clean)
OUT_TABLE = "axahealth_dataplatform_pd_lab.axahealth_data_scientist_analytics.journeylenz_full_25_26_out_data_unflattened"

# # Saving small/medium spark_dfs
# (
#     spark_df.write
#     .format("delta")
#     .mode("overwrite")
#     .option("overwriteSchema", "true")
#     .saveAsTable(OUT_TABLE)
# )

# Appending chunk of larger spark_df to df
spark_df.write.mode("append").format("delta").saveAsTable(OUT_TABLE)

In [0]:
import ast
import pandas as pd

df = df_out_clean.copy()

def parse_topics(x):
    if isinstance(x, list):
        return x
    if x is None or pd.isna(x):
        return []
    if isinstance(x, str):
        s = x.strip()
        if s in ["", "null", "None", "nan"]:
            return []
        try:
            return json.loads(s)
        except Exception:
            pass
        try:
            return ast.literal_eval(s)
        except Exception:
            return []
    return []

df["topics_struct"] = df["topics_struct"].apply(parse_topics)

rows = []

for _, row in df.iterrows():
    claim_number = row.get("ClaimNumber")
    membership_number = row.get("MembershipNumber")
    conversation_id = row.get("ConversationId")

    for bucket_item in row["topics_struct"]:
        time_of_contact = bucket_item.get("time_of_contact")
        bucket = bucket_item.get("bucket")
        sub_bucket = bucket_item.get("sub_bucket")
        coverage_pct = bucket_item.get("coverage_pct")
        talk_time_mins = bucket_item.get("talk_time_mins")

        for topic_item in bucket_item.get("topics", []):
            topic = topic_item.get("topic")
            reasons = topic_item.get("reasons", [])

            if not reasons:
                reasons = [{}]

            for reason in reasons:
                rows.append({
                    "ClaimNumber": claim_number,
                    "MembershipNumber": membership_number,
                    "ConversationId": conversation_id,
                    "time_of_contact": time_of_contact,
                    "bucket": bucket,
                    "sub_bucket": sub_bucket,
                    "coverage_pct": coverage_pct,
                    "talk_time_mins": talk_time_mins,

                    "topic": topic,
                    "sentiment": bucket_item.get("sentiment"),
                    "resolved": bucket_item.get("resolved"),
                    "evidence_score": reason.get("evidence_score"),
                    "suggestion": reason.get("suggestion"),

                    "description": reason.get("description"),
                    "evidence": reason.get("evidence")
                })
topics_df = pd.DataFrame(rows) 

In [0]:
topics_df = topics_df.rename(
    columns={
        "bucket": "topic",
        "sub_bucket": "sub_topic",
        "topic": "topic_llm"
    }
)

In [0]:
#write flattened output
spark_df = spark.createDataFrame(topics_df)
OUT_TABLE = "axahealth_dataplatform_pd_lab.axahealth_data_scientist_analytics.journeylenz_full_25_26_out_data_flattened"

# # Saving small/medium spark_dfs
# (
#     spark_df.write
#     .format("delta")
#     .mode("overwrite")
#     .option("overwriteSchema", "true")
#     .saveAsTable(OUT_TABLE)
# )

# Appending chunk of larger spark_df to df
spark_df.write.mode("append").format("delta").saveAsTable(OUT_TABLE)

# Build from checkpoint

In [0]:
CHECKPOINT_TABLE = (
    "axahealth_dataplatform_pd_lab.axahealth_data_scientist_analytics.journeylenz_output_checkpoint_full_data_run"
)

def parse_analysis(value):
    if isinstance(value, dict):
        return value
    
    if value is None:
        return {}
    
    if isinstance(value, str):
        value = value.strip()

        if not value:
            return {}
        
        try:
            return json.loads(value)
        except json.JSONDecodeError:
            try:
                return ast.literal_eval(value)
            except (ValueError, SyntaxError):
                return {}
    return {}

df_sp_checkpoint = spark.table(CHECKPOINT_TABLE)
df_out = df_sp_checkpoint.toPandas()
df_out["analysis"] = df_out["analysis"].apply(parse_analysis)

#recreate post-processing columns
df_out["topics_struct"] = df_out.apply(
    lambda row: build_nested_from_analysis(
        row["analysis"],
        float(row.get("ConversationTimeSecs", 0) or 0),
        str(row.get("ConversationContent", "") or ""),
        str(row.get("ContactChannel", "") or ""),
    ),
    axis=1,
)
df_out["overall_sentiment"] = (
    df_out["topics_struct"].apply(weighted_sentiment)
)
df_out["first_bucket"] = (
    df_out["topics_struct"].apply(get_first_bucket)
)
df_out["dominant_bucket"] = (
    df_out["topics_struct"].apply(get_dominant_bucket)
)
display(df_out)

In [0]:
df_out["topics_pretty"] = df_out["topics_struct"].apply(
    lambda x: json.dumps(x, indent=2)
)
display(df_out[["MembershipNumber", "ClaimNumber", "ConversationId", "ConversationContent", "prompt_tokens", "completion_tokens", "total_tokens", "topics_pretty", "overall_sentiment", "first_bucket", "dominant_bucket"]])

In [0]:
df_out_clean = df_out.drop(columns=["analysis"])

In [0]:
object_cols = [
    "error",
    "topics_struct",
    "first_bucket",
    "dominant_bucket",
    "topics_pretty"
]
for c in object_cols:
    df_out_clean[c] = df_out_clean[c].astype(str)

In [0]:
#write intial output from checkpoint
spark_df = spark.createDataFrame(df_out_clean)
OUT_TABLE = "axahealth_dataplatform_pd_lab.axahealth_data_scientist_analytics.journeylenz_full_25_26_out_data_unflattened"

# # Saving small/medium spark_dfs
# (
#     spark_df.write
#     .format("delta")
#     .mode("overwrite")
#     .option("overwriteSchema", "true")
#     .saveAsTable(OUT_TABLE)
# )

# Appending chunk of larger spark_df to df
spark_df.write.mode("append").format("delta").saveAsTable(OUT_TABLE)

In [0]:
import ast
import pandas as pd

df = df_out_clean.copy()

def parse_topics(x):
    if isinstance(x, list):
        return x
    if x is None or pd.isna(x):
        return []
    if isinstance(x, str):
        s = x.strip()
        if s in ["", "null", "None", "nan"]:
            return []
        try:
            return json.loads(s)
        except Exception:
            pass
        try:
            return ast.literal_eval(s)
        except Exception:
            return []
    return []

df["topics_struct"] = df["topics_struct"].apply(parse_topics)

rows = []

for _, row in df.iterrows():
    claim_number = row.get("ClaimNumber")
    membership_number = row.get("MembershipNumber")
    conversation_id = row.get("ConversationId")

    for bucket_item in row["topics_struct"]:
        time_of_contact = bucket_item.get("time_of_contact")
        bucket = bucket_item.get("bucket")
        sub_bucket = bucket_item.get("sub_bucket")
        coverage_pct = bucket_item.get("coverage_pct")
        talk_time_mins = bucket_item.get("talk_time_mins")

        for topic_item in bucket_item.get("topics", []):
            topic = topic_item.get("topic")
            reasons = topic_item.get("reasons", [])

            if not reasons:
                reasons = [{}]

            for reason in reasons:
                rows.append({
                    "ClaimNumber": claim_number,
                    "MembershipNumber": membership_number,
                    "ConversationId": conversation_id,
                    "time_of_contact": time_of_contact,
                    "bucket": bucket,
                    "sub_bucket": sub_bucket,
                    "coverage_pct": coverage_pct,
                    "talk_time_mins": talk_time_mins,

                    "topic": topic,
                    "sentiment": bucket_item.get("sentiment"),
                    "resolved": bucket_item.get("resolved"),
                    "evidence_score": reason.get("evidence_score"),
                    "suggestion": reason.get("suggestion"),

                    "description": reason.get("description"),
                    "evidence": reason.get("evidence")
                })
topics_df = pd.DataFrame(rows)
display(topics_df)

In [0]:
topics_df = topics_df.rename(
    columns={
        "bucket": "topic",
        "sub_bucket": "sub_topic",
        "topic": "topic_llm"
    }
)

In [0]:
#write flattened output from checkpoint
spark_df = spark.createDataFrame(topics_df)
OUT_TABLE = "axahealth_dataplatform_pd_lab.axahealth_data_scientist_analytics.journeylenz_full_25_26_out_data_flattened"
# # Saving small/medium spark_dfs
# (
#     spark_df.write
#     .format("delta")
#     .mode("overwrite")
#     .option("overwriteSchema", "true")
#     .saveAsTable(OUT_TABLE)
# )

# Appending chunk of larger spark_df to df
spark_df.write.mode("append").format("delta").saveAsTable(OUT_TABLE)

# Intitial analysis. this is for my own refernce/interest only and is not needed by anyone else

In [0]:
df = spark.table(
    "axahealth_dataplatform_pd_lab.axahealth_data_scientist_analytics.journeylenz_new_format_testing_flattened"
).toPandas()

In [0]:
df.head()

In [0]:
sentiment_analysis = (
    topics_df
    .groupby("sub_topic")
    .agg(
        num_mentions=("sub_topic", "count"),
        avg_sentiment=("sentiment", "mean"),
        avg_evidence=("evidence_score", "mean")
    )
    .sort_values("num_mentions", ascending=False)
)
sentiment_analysis.head(10)

In [0]:
resolution_analysis = (
    topics_df
    .groupby("sub_topic")
    .agg(
        mentions=("sub_topic", "count"),
        resolution_rate=("resolved", "mean")
    )
    .sort_values("mentions", ascending=False)
)
resolution_analysis.head(20)

In [0]:
sub_topic_analysis = (
    topics_df
    .groupby("sub_topic")
    .agg(
        mentions=("sub_topic", "count"),
        avg_sentiment=("sentiment", "mean"),
        resolution_rate=("resolved", "mean"),
        avg_coverage=("coverage_pct", "mean")
    )
    .sort_values("mentions", ascending=False)
)
sub_topic_analysis